# Lithuanian speech-to-text — Parakeet (Kmynas) on Colab

Transcribes Lithuanian audio to a punctuated, speaker-labelled transcript
plus `.srt` / `.vtt` / `.json`, using a NeMo Parakeet-TDT fine-tune.

Faster than the Whisper pipeline (`colab_quickstart.ipynb` in the paprika repo) — real-time
factor around 0.03 on a T4, so a 35-minute recording decodes in about a
minute — and the model punctuates by itself.

**Set the runtime to GPU**: Runtime → Change runtime type → T4 GPU.
CPU works but is roughly 30× slower.

Repo: https://github.com/kristijonasatpro/kmynas


## 1. Install

NeMo is a large install — budget 5–10 minutes. It pins versions that
conflict with the Whisper pipeline, which is why that one lives in a
separate environment locally; on Colab just don't run both notebooks in
the same session.


In [ ]:
import os
if not os.path.isdir('/content/kmynas'):
    !git clone -q https://github.com/kristijonasatpro/kmynas
%cd /content/kmynas
!pip install -q 'nemo_toolkit[asr]' silero-vad soundfile sherpa-onnx

import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '— good to go')
else:
    print('NO GPU. Runtime > Change runtime type > T4 GPU, then rerun.')


## 2. Get the model

v3 is public, so this needs no token and no Hugging Face account. About
2.5 GB, cached for the rest of the session.

To compare against another checkpoint, change `REPO` and `FILE`. If that one
is private, uncomment the two `HF_TOKEN` lines and add the secret in Colab
(key icon, left sidebar).


In [ ]:
from huggingface_hub import hf_hub_download

REPO = 'kristijonas/kmynas-parakeet-lt-v3'
FILE = 'kmynas-parakeet-lt-v3.nemo'

# Only needed if you point REPO at a private checkpoint of your own:
# import os; from google.colab import userdata
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

MODEL = hf_hub_download(REPO, FILE)
print('model at', MODEL)


## 3. Add your audio

Run the cell and pick a file (m4a, mp3, wav, mp4, mov — anything ffmpeg
reads), or mount Drive instead:

```python
from google.colab import drive; drive.mount('/content/drive')
AUDIO = '/content/drive/MyDrive/your-recording.m4a'
```


In [ ]:
from google.colab import files
up = files.upload()
AUDIO = list(up)[0]
print('using', AUDIO)


## 4. Transcribe

`--speakers N` when you know how many people are talking — it is much more
reliable than letting the clustering threshold guess. `--no-diar` skips
speaker labels and is faster.

Leave `--dtype` alone: fp16 halves device memory, decodes about 15% faster,
and differs from fp32 on 0.39% of words with neither side consistently
better. Only v2 checkpoints need `--dtype fp32`.


In [ ]:
SPEAKERS = 2        # how many people are talking. Set it — guessing is unreliable.

!python transcribe_kmynas.py "$AUDIO" \
    --model "$MODEL" \
    --speakers $SPEAKERS \
    --lexicon lexicon.tsv \
    --review


## 5. Read it, then download

The first block is the transcript. The second is the review list — the spans
the model was least sure about, worst first. On this model every error found
by reading a transcript end to end scored below the threshold, so start there
rather than reading all of it. It will not catch a *confident* mistake, so it
narrows the read rather than replacing it.


In [ ]:
import pathlib
stem = pathlib.Path(AUDIO).with_suffix('')

print(pathlib.Path(f'{stem}.txt').read_text()[:4000])

review = pathlib.Path(f'{stem}.review.txt')
if review.exists():
    print('\n' + '=' * 70)
    print('WHERE TO CHECK IT — worst 25 spans by confidence\n')
    rows = [l for l in review.read_text().splitlines() if l.startswith('[')]
    for line in sorted(rows, key=lambda l: float(l.split()[1]))[:25]:
        print(line)


In [ ]:
for ext in ('txt', 'review.txt', 'srt', 'vtt', 'json'):
    p = pathlib.Path(f'{stem}.{ext}')
    if p.exists():
        files.download(str(p))


## Tuning, if the output looks wrong

| symptom | flag to reach for |
|---|---|
| short consonant-cluster junk (`chl`, `žl`) on breath or laughter | raise `--min-confidence` toward `0.99` |
| real quiet words disappearing | lower `--min-confidence`, or `0` to disable |
| words broken across block joins | you are probably in `--no-vad` mode; drop the flag |
| out of GPU memory | lower `--vad-target` (60 → 30); check you are not on `--dtype fp32` |
| the same borrowed word spelled several ways | edit `lexicon.tsv` |
| you want to know *where* the errors are | it is printed above, and in `<name>.review.txt` |
| decoding feels slow on the T4 | `--batch-size 4` (default 1 is the safe setting, not the fast one) |
| speaker labels error out | `--no-diar`; the transcript is unaffected apart from the labels |
| speaker labels obviously wrong | pass `--speakers N`, or `--no-diar` |

Block length is the memory knob and attention cost grows with its square.
A T4's 15 GB handles 60 s blocks comfortably. Whole-file single-pass
decoding (no blocks) works up to roughly 5 minutes of audio and runs out
of memory well before 10.

The design reasoning behind each of these — including several fixes that
were tried and measured *worse* — is in the docstrings of
`transcribe_kmynas.py` and `chunk_longform.py`.
